In [ ]:
import os
import glob
import re
import random
import gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import nltk
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

from ai4bharat.transliteration import XlitEngine

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

nltk.download('wordnet', quiet=True)

In [ ]:
data_dir = "backtransliteration"
out_dir = "outputs_backtransliteration/"

os.makedirs(out_dir, exist_ok=True)

model_name = "google/madlad400-3b-mt"
device = "cuda" if torch.cuda.is_available() else "cpu"
output_path = f"{out_dir}{model_name.split('/')[1]}_mt_evaluation_backtransliteration_results_I.csv"

In [ ]:
BATCH_SIZE = 64 

LANG_MAP = {
    "Nagamese": {"madlad": "as",  "xlit": "as"},
    "Nyishi":   {"madlad": "brx", "xlit": "brx"},  
    "Tagin":    {"madlad": "brx", "xlit": "brx"},
    "Kokborok": {"madlad": "brx", "xlit": "brx"},
    "Khasi":    {"madlad": "kha", "xlit": None},
    "Karbi":    {"madlad": "lus", "xlit": "lus"},
    "Mizo":     {"madlad": "lus", "xlit": None}
    
}

bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    use_safetensors=True, 
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

if hasattr(torch, "compile") and device == "cuda":
    print("Compiling model for faster inference execution...")
    model = torch.compile(model)

In [ ]:
def determine_lang_info(filename):
    for lang, config in LANG_MAP.items():
        if lang.lower() in filename.lower():
            return lang, config
    return None, None

def detect_columns(df, filename):
    cols = [str(c).strip() for c in df.columns]    
    if len(cols) < 2:
        raise ValueError("File does not contain at least 2 columns.")
    df.columns = cols
    en_indicators = ['english', 'en', 'eng', 'English']
    en_col = next((col for col in cols if col.lower() in en_indicators), None)
    if not en_col:
        en_col = cols[1] 
    src_col = [c for c in cols if c != en_col][0]
    return src_col, en_col

def run_transliteration(texts, xlit_code, lang_name):
    if not xlit_code:
        print(f"-> {lang_name} naturally utilizes Roman script. Skipping back-transliteration.")
        return texts
        
    print(f"-> Converting Romanized {lang_name} into native script variant ({xlit_code})...")
    try:
        e = XlitEngine(xlit_code, beam_width=4, rescore=True, src_script_type="roman")
        transliterated_texts = []
        xlit_batch_size = 256
        
        for i in range(0, len(texts), xlit_batch_size):
            batch = [str(t) for t in texts[i : i + xlit_batch_size]]
            try:
                out = e.translit_batch(batch)
                if isinstance(out, dict) and xlit_code in out:
                    transliterated_texts.extend(out[xlit_code])
                else:
                    transliterated_texts.extend(batch)
            except Exception:
                transliterated_texts.extend(batch)
                
        del e
        gc.collect()
        return transliterated_texts
    except Exception as ex:
        print(f" [Error] Transliteration engine failed for code '{xlit_code}': {ex}. Using original raw strings.")
        return texts  
        
def translate_batch_generator(texts, tgt_code="en", batch_size=BATCH_SIZE):
    prefix = f"<2{tgt_code}> "
    for i in range(0, len(texts), batch_size):
        batch = [prefix + str(text) for text in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.inference_mode():
            generated_tokens = model.generate(**inputs, max_length=256, num_beams=1, use_cache=True, do_sample=False)
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        yield decoded
        
def save_incremental_results(results_dict, out_path):
    if not results_dict:
        return
    res_df = pd.DataFrame.from_dict(results_dict, orient='index')
    res_df.to_csv(out_path)
    print(f"\n--- Live metrics written to: {out_path} ---")

In [ ]:
all_files = glob.glob(os.path.join(data_dir, "*.csv")) + glob.glob(os.path.join(data_dir, "*.xlsx"))
language_results = {}

for file_path in all_files:
    filename = os.path.basename(file_path)
    lang_name, config = determine_lang_info(filename)
    
    if not lang_name:
        print(f"Skipping {filename}: Could not determine target language grouping.")
        continue
        
    src_code = config["madlad"]
    xlit_code = config["xlit"]
    
    if not src_code:
        print(f"Skipping {filename}: Language '{lang_name}' is not mapped to a valid target language.")
        continue   
        
    print(f"\n" + "-"*40)
    print(f"Processing Generation and Evaluation Metrics for: {lang_name} ({filename})")
    print("-"*40)   
    
    try:
        df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)     
        src_col, en_col = detect_columns(df, filename)
        df = df[[src_col, en_col]].dropna().astype(str)
        
        raw_src_sentences = df[src_col].tolist()
        ref_sentences = df[en_col].tolist()
        
        native_sentences = run_transliteration(raw_src_sentences, xlit_code, lang_name)
        
        print(f"Translating {len(native_sentences)} sentences using MADLAD code context '{src_code}'...")
        hyp_sentences = []
        
        for batch_hyp in tqdm(translate_batch_generator(native_sentences, tgt_code="en", batch_size=BATCH_SIZE), 
                              total=(len(native_sentences) + (BATCH_SIZE - 1)) // BATCH_SIZE, desc="MADLAD Inference"):
            hyp_sentences.extend(batch_hyp)

        prediction_df = df.copy()
        prediction_df["native"] = native_sentences
        prediction_df["Prediction"] = hyp_sentences
        
        prediction_output_path = f"{os.path.splitext(filename)[0]}_backtransliterated_predictions.csv"
        prediction_df.to_csv(prediction_output_path, index=False)
        print(f"Predictions saved to: {prediction_output_path}")  
        
        # Step 3: Compute Evaluation Block metrics
        bleu_ref = [[ref] for ref in ref_sentences]
        scores = {"BLEU": None, "chrF": None, "ROUGE-1": None, "ROUGE-L": None, "METEOR": None, "BERTScore-F1": None}
        
        try: scores["BLEU"] = round(bleu_metric.compute(predictions=hyp_sentences, references=bleu_ref)['bleu'] * 100, 2)
        except Exception as me: print(f" [Error] BLEU fail: {me}")
        try: scores["chrF"] = round(chrf_metric.compute(predictions=hyp_sentences, references=bleu_ref)['score'], 2)
        except Exception as me: print(f" [Error] chrF fail: {me}")
        try:
            rouge = rouge_metric.compute(predictions=hyp_sentences, references=ref_sentences)
            scores["ROUGE-1"], scores["ROUGE-L"] = round(rouge['rouge1'] * 100, 2), round(rouge['rougeL'] * 100, 2)
        except Exception as me: print(f" [Error] ROUGE fail: {me}")
        try: scores["METEOR"] = round(meteor_metric.compute(predictions=hyp_sentences, references=ref_sentences)['meteor'] * 100, 2)
        except Exception as me: print(f" [Error] METEOR fail: {me}")
        try:
            bertscore = bertscore_metric.compute(predictions=hyp_sentences, references=ref_sentences, lang="en", batch_size=BATCH_SIZE, device=device)
            scores["BERTScore-F1"] = round(sum(bertscore['f1']) / len(bertscore['f1']) * 100, 2)
        except Exception as me: print(f" [Error] BERTScore fail: {me}")
            
        language_results[lang_name] = scores
        save_incremental_results(language_results, output_path)
        
    except Exception as e:
        print(f"  Critical pipeline error occurred processing {filename}: {e}")
    finally:
        gc.collect()
        if device == "cuda": 
            torch.cuda.empty_cache()

if language_results:
    final_df = pd.DataFrame.from_dict(language_results, orient='index')
    calc_df = final_df.drop('OVERALL (Macro Avg)', errors='ignore')
    overall_avg = calc_df.mean(numeric_only=True).round(2)
    final_df.loc['OVERALL (Macro Avg)'] = overall_avg  
    print("\n" + "="*60)
    print(" FINAL EVALUATION SUMMARY ")
    print("="*60)
    print(final_df.to_markdown())
else:
    print("No languages were successfully processed.")